# ResNet-18 on CIFAR-10 – From Scratch with PyTorch

**Model:** ResNet-18 (CIFAR-10 adapted) – implemented from scratch  
**Optimizer:** Custom Adam (no `torch.optim.Adam`)  
**Loss:** Custom CrossEntropyLoss (no `nn.CrossEntropyLoss`)  
**Dataset:** CIFAR-10 – 60 000 images, 10 classes, 32×32 px

---
## CIFAR-10 classes
airplane · automobile · bird · cat · deer · dog · frog · horse · ship · truck

## ResNet-18 Architecture
```
stem   : Conv(3→64, 3×3, s=1) → BN → ReLU
layer1 : 2× BasicBlock(64→64,   s=1)
layer2 : 2× BasicBlock(64→128,  s=2)
layer3 : 2× BasicBlock(128→256, s=2)
layer4 : 2× BasicBlock(256→512, s=2)
pool   : AdaptiveAvgPool2d(1×1)
fc     : Linear(512→10)
```
Each BasicBlock: Conv→BN→ReLU→Conv→BN + skip (projection if dims change) → ReLU

## 1. Imports & device

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

from classes import (
    CIFAR10Dataset, ResNet18,
    CrossEntropyLoss, Adam,
    CIFAR10_CLASSES,
)
from utils import (
    train_loop, test_loop,
    plot_loss_curves, plot_accuracy_curve,
    plot_confusion_matrix, show_sample_predictions,
    count_parameters,
)

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch: {torch.__version__}  |  Device: {device}')

## 2. Verify custom loss and optimiser
Quick sanity check before training: compare our `CrossEntropyLoss` to PyTorch's.

In [ ]:
import torch.nn as nn

torch.manual_seed(0)
logits  = torch.randn(8, 10)
targets = torch.randint(0, 10, (8,))

our_loss = CrossEntropyLoss()(logits, targets).item()
ref_loss = nn.CrossEntropyLoss()(logits, targets).item()

print(f'Custom loss : {our_loss:.6f}')
print(f'PyTorch ref : {ref_loss:.6f}')
print(f'Difference  : {abs(our_loss - ref_loss):.2e}  ← should be ~0')

## 3. Data

In [ ]:
DATA_ROOT = './data'

train_dataset = CIFAR10Dataset(DATA_ROOT, train=True,  augment=True)
test_dataset  = CIFAR10Dataset(DATA_ROOT, train=False, augment=False)

train_loader  = DataLoader(train_dataset, batch_size=128, shuffle=True,  num_workers=2)
test_loader   = DataLoader(test_dataset,  batch_size=128, shuffle=False, num_workers=2)

print(f'Train: {len(train_dataset):,}  |  Test: {len(test_dataset):,}')
print(f'Classes: {CIFAR10_CLASSES}')

In [ ]:
# Visualise a sample batch (unnormalise for display)
mean = np.array([0.4914, 0.4822, 0.4465])
std  = np.array([0.2470, 0.2435, 0.2616])

X_show, y_show = next(iter(DataLoader(test_dataset, batch_size=20, shuffle=True)))
fig, axes = plt.subplots(2, 10, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    img = X_show[i].permute(1, 2, 0).numpy() * std + mean
    ax.imshow(np.clip(img, 0, 1))
    ax.set_title(CIFAR10_CLASSES[y_show[i].item()], fontsize=8)
    ax.axis('off')
plt.suptitle('CIFAR-10 sample images')
plt.tight_layout()
plt.show()

## 4. Model overview

In [ ]:
torch.manual_seed(RANDOM_SEED)
model = ResNet18(num_classes=10).to(device)
print(model)
print(f'\nTotal trainable parameters: {count_parameters(model):,}')

# Forward pass shape check
dummy = torch.zeros(4, 3, 32, 32).to(device)
print(f'Input  shape: {dummy.shape}')
print(f'Output shape: {model(dummy).shape}')

## 5. Baseline training (Adam lr=1e-3, 30 epochs)

In [ ]:
EPOCHS = 30
LR     = 1e-3

torch.manual_seed(RANDOM_SEED)
model     = ResNet18(num_classes=10).to(device)
loss_fn   = CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=LR, weight_decay=1e-4)

train_losses, test_losses, accuracies = [], [], []

for epoch in range(1, EPOCHS + 1):
    tr_loss       = train_loop(train_loader, model, loss_fn, optimizer, device)
    te_loss, acc  = test_loop(test_loader,   model, loss_fn,            device)

    train_losses.append(tr_loss)
    test_losses.append(te_loss)
    accuracies.append(acc)

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:>3d}/{EPOCHS}  '
              f'Train: {tr_loss:.4f}  '
              f'Test: {te_loss:.4f}  '
              f'Acc: {acc:.1f}%')

print(f'\nBest accuracy: {max(accuracies):.1f}%  (epoch {accuracies.index(max(accuracies))+1})')

In [ ]:
plot_loss_curves(train_losses, test_losses, title='Baseline – Loss curves (Adam lr=1e-3)')
plot_accuracy_curve(accuracies, title='Baseline – Test accuracy')

In [ ]:
show_sample_predictions(model, test_loader, device)

## 6. Experiment: Learning rate

In [ ]:
lr_results = {}
EXP_EPOCHS = 15

for lr in [1e-2, 1e-3, 5e-4, 1e-4]:
    torch.manual_seed(RANDOM_SEED)
    m   = ResNet18(num_classes=10).to(device)
    opt = Adam(m.parameters(), lr=lr, weight_decay=1e-4)
    lf  = CrossEntropyLoss()

    accs = []
    for _ in range(EXP_EPOCHS):
        train_loop(train_loader, m, lf, opt, device)
        _, acc = test_loop(test_loader, m, lf, device)
        accs.append(acc)

    lr_results[lr] = accs
    print(f'lr={lr:.0e}  best={max(accs):.1f}%  final={accs[-1]:.1f}%')

plt.figure(figsize=(9, 4))
for lr, accs in lr_results.items():
    plt.plot(range(1, EXP_EPOCHS + 1), accs, label=f'lr={lr:.0e}')
plt.xlabel('Epoch')
plt.ylabel('Test accuracy (%)')
plt.title('Learning rate – Adam')
plt.legend()
plt.tight_layout()
plt.show()

## 7. Experiment: Adam vs SGD

In [ ]:
opt_results = {}

for name, build_opt in [
    ('Custom Adam (1e-3)', lambda p: Adam(p, lr=1e-3, weight_decay=1e-4)),
    ('Custom Adam (5e-4)', lambda p: Adam(p, lr=5e-4, weight_decay=1e-4)),
    ('SGD+momentum (1e-2)', lambda p: torch.optim.SGD(p, lr=1e-2, momentum=0.9, weight_decay=1e-4)),
    ('SGD+momentum (1e-1)', lambda p: torch.optim.SGD(p, lr=1e-1, momentum=0.9, weight_decay=1e-4)),
]:
    torch.manual_seed(RANDOM_SEED)
    m   = ResNet18(num_classes=10).to(device)
    opt = build_opt(m.parameters())
    lf  = CrossEntropyLoss()

    accs = []
    for _ in range(EXP_EPOCHS):
        train_loop(train_loader, m, lf, opt, device)
        _, acc = test_loop(test_loader, m, lf, device)
        accs.append(acc)

    opt_results[name] = accs
    print(f'{name:<30s}  best={max(accs):.1f}%  final={accs[-1]:.1f}%')

plt.figure(figsize=(9, 4))
for name, accs in opt_results.items():
    ls = '-' if 'Adam' in name else '--'
    plt.plot(range(1, EXP_EPOCHS + 1), accs, linestyle=ls, label=name)
plt.xlabel('Epoch')
plt.ylabel('Test accuracy (%)')
plt.title('Custom Adam vs SGD')
plt.legend()
plt.tight_layout()
plt.show()

## 8. Experiment: Augmentation vs no augmentation

In [ ]:
aug_results = {}

for aug in [True, False]:
    ds  = CIFAR10Dataset(DATA_ROOT, train=True, augment=aug)
    dl  = DataLoader(ds, batch_size=128, shuffle=True, num_workers=2)

    torch.manual_seed(RANDOM_SEED)
    m   = ResNet18(num_classes=10).to(device)
    opt = Adam(m.parameters(), lr=1e-3, weight_decay=1e-4)
    lf  = CrossEntropyLoss()

    accs = []
    for _ in range(EXP_EPOCHS):
        train_loop(dl, m, lf, opt, device)
        _, acc = test_loop(test_loader, m, lf, device)
        accs.append(acc)

    label = 'With augmentation' if aug else 'No augmentation'
    aug_results[label] = accs
    print(f'{label:<22s}  best={max(accs):.1f}%  final={accs[-1]:.1f}%')

plt.figure(figsize=(9, 4))
for label, accs in aug_results.items():
    plt.plot(range(1, EXP_EPOCHS + 1), accs, label=label)
plt.xlabel('Epoch')
plt.ylabel('Test accuracy (%)')
plt.title('Effect of data augmentation')
plt.legend()
plt.tight_layout()
plt.show()

## 9. Final model – best config

In [ ]:
BEST_EPOCHS = 50
BEST_LR     = 1e-3

torch.manual_seed(RANDOM_SEED)
best_model = ResNet18(num_classes=10).to(device)
loss_fn    = CrossEntropyLoss()
optimizer  = Adam(best_model.parameters(), lr=BEST_LR, weight_decay=1e-4)

tr_losses, te_losses, accs = [], [], []

for epoch in range(1, BEST_EPOCHS + 1):
    trl       = train_loop(train_loader, best_model, loss_fn, optimizer, device)
    tel, acc  = test_loop(test_loader,   best_model, loss_fn,            device)
    tr_losses.append(trl)
    te_losses.append(tel)
    accs.append(acc)
    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:>3d}/{BEST_EPOCHS}  Acc: {acc:.1f}%')

print(f'\nBest accuracy: {max(accs):.1f}%  (epoch {accs.index(max(accs))+1})')

plot_loss_curves(tr_losses, te_losses, title='Final model – Loss curves')
plot_accuracy_curve(accs, title='Final model – Test accuracy')

In [ ]:
plot_confusion_matrix(best_model, test_loader, device)
show_sample_predictions(best_model, test_loader, device)

## 10. Save

In [ ]:
torch.save({'model_state': best_model.state_dict()}, 'resnet_model.pth')
print('Model saved to resnet_model.pth')
print('Run app.py to classify your own images.')